# Predictive Coding Network — Advanced MNIST

Custom training loop with optimizer selection and progress monitoring.

Architecture:

```
pixels(784) ──→ h1(256) ──→ h2(128) ──→ h3(64) ──→ class(10)
 Identity       Sigmoid     Sigmoid     Sigmoid     Sigmoid
```

## Imports & JAX Setup

In [ ]:
import os
import jax
import jax.numpy as jnp
import time
import optax

from fabricpc.nodes import Linear
from fabricpc.core.topology import Edge
from fabricpc.graph_assembly import TaskMap, graph
from fabricpc.graph_initialization import initialize_params, FeedforwardStateInit
from fabricpc.core.activations import IdentityActivation, SigmoidActivation
from fabricpc.core.energy import GaussianEnergy
from fabricpc.core.initializers import NormalInitializer
from fabricpc.core.inference import InferenceSGD
from fabricpc.training import train_step, evaluate_pcn
from fabricpc.training.natural_gradients import (
    scale_by_natural_gradient_diag,
    scale_by_natural_gradient_layerwise,
)
from fabricpc.utils.data.dataloader import MnistLoader
from fabricpc import setup_jax

setup_jax()
jax.config.update("jax_default_prng_impl", "threefry2x32")

## Define the Network

In [ ]:
pixels = Linear(shape=(784,), activation=IdentityActivation(), name="pixels")
h1 = Linear(
    shape=(256,),
    activation=SigmoidActivation(),
    energy=GaussianEnergy(precision=1.0),
    weight_init=NormalInitializer(mean=0.0, std=0.05),
    name="h1",
)
h2 = Linear(
    shape=(128,),
    activation=SigmoidActivation(),
    energy=GaussianEnergy(precision=1.0),
    weight_init=NormalInitializer(mean=0.0, std=0.05),
    name="h2",
)
h3 = Linear(
    shape=(64,),
    activation=SigmoidActivation(),
    energy=GaussianEnergy(precision=1.0),
    weight_init=NormalInitializer(mean=0.0, std=0.05),
    name="h3",
)
class_node = Linear(
    shape=(10,),
    activation=SigmoidActivation(),
    energy=GaussianEnergy(precision=1.0),
    weight_init=NormalInitializer(mean=0.0, std=0.05),
    name="class",
)

structure = graph(
    nodes=[pixels, h1, h2, h3, class_node],
    edges=[
        Edge(source=pixels, target=h1.slot("in")),
        Edge(source=h1, target=h2.slot("in")),
        Edge(source=h2, target=h3.slot("in")),
        Edge(source=h3, target=class_node.slot("in")),
    ],
    task_map=TaskMap(x=pixels, y=class_node),
    graph_state_initializer=FeedforwardStateInit(),
    inference=InferenceSGD(eta_infer=0.05, infer_steps=20),
)

## Hyperparameters & Optimizer

In [ ]:
train_config = {}
batch_size = 200
num_epochs = 10

OPTIMIZER_PRESETS = {
    "adam": lambda: optax.chain(optax.adam(0.001)),
    "adamw": lambda: optax.adamw(0.001, weight_decay=0.1),
    "sgd": lambda: optax.chain(
        optax.add_decayed_weights(0.1), optax.sgd(0.01, momentum=0.9)
    ),
    "ngd_diag": lambda: optax.chain(
        optax.add_decayed_weights(0.1),
        scale_by_natural_gradient_diag(fisher_decay=0.95, damping=1e-3),
        optax.scale(-0.0003),
    ),
    "ngd_layerwise": lambda: optax.chain(
        optax.add_decayed_weights(0.1),
        scale_by_natural_gradient_layerwise(fisher_decay=0.95, damping=1e-3),
        optax.scale(-0.001),
    ),
}


def get_optimizer(name: str) -> optax.GradientTransformation:
    """Return optimizer preset by name."""
    key = name.lower()
    if key not in OPTIMIZER_PRESETS:
        valid = ", ".join(OPTIMIZER_PRESETS.keys())
        raise ValueError(f"unknown optimizer '{name}'. valid choices: {valid}")
    return OPTIMIZER_PRESETS[key]()


optimizer_name = "adamw"  # Change to: adam, adamw, sgd, ngd_diag, ngd_layerwise
optimizer = get_optimizer(optimizer_name)

## Initialize Parameters & Data Loaders

In [ ]:
master_rng_key = jax.random.PRNGKey(42)
graph_key, train_key, eval_key = jax.random.split(master_rng_key, 3)

params = initialize_params(structure, graph_key)
num_params = sum(p.size for p in jax.tree_util.tree_leaves(params))

print(
    f"\n{len(structure.nodes)} nodes, {len(structure.edges)} edges, "
    f"{num_params:,} parameters"
)
print(f"Optimizer: {optimizer_name}")

train_loader = MnistLoader(
    "train", batch_size=batch_size, tensor_format="flat", shuffle=True, seed=42
)
test_loader = MnistLoader(
    "test", batch_size=batch_size, tensor_format="flat", shuffle=False
)

## Train

In [ ]:
opt_state = optimizer.init(params)

jit_train_step = jax.jit(
    lambda p, o, b, k: train_step(p, o, b, structure, optimizer, k)
)

print(f"\nTraining for {num_epochs} epochs (JIT compilation on first batch)...\n")

best_accuracy = 0.0
training_history = []

num_batches = len(train_loader)
all_rng_keys = jax.random.split(train_key, num_epochs * num_batches)
all_rng_keys = all_rng_keys.reshape((num_epochs, num_batches, 2))

for epoch in range(num_epochs):
    epoch_start = time.time()
    epoch_energies = []

    for batch_idx, (x, y) in enumerate(train_loader):
        batch = {"x": jnp.array(x), "y": y}

        params, opt_state, energy, _ = jit_train_step(
            params, opt_state, batch, all_rng_keys[epoch, batch_idx]
        )
        epoch_energies.append(float(energy))

        if (batch_idx + 1) % 100 == 0:
            avg_energy = sum(epoch_energies[-100:]) / len(epoch_energies[-100:])
            print(
                f"  Epoch {epoch+1}/{num_epochs}, "
                f"Batch {batch_idx+1}/{num_batches}, "
                f"energy: {avg_energy:.4f}"
            )

    epoch_time = time.time() - epoch_start
    avg_energy = sum(epoch_energies) / len(epoch_energies)

    epoch_eval_key, eval_key = jax.random.split(eval_key)
    metrics = evaluate_pcn(
        params, structure, test_loader, train_config, epoch_eval_key
    )
    accuracy = metrics["accuracy"] * 100

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_params = params

    print(
        f"  Epoch {epoch+1}/{num_epochs} - "
        f"energy: {avg_energy:.4f}, Accuracy: {accuracy:.2f}%, "
        f"Time: {epoch_time:.1f}s"
    )

    training_history.append(
        {
            "epoch": epoch + 1,
            "energy": avg_energy,
            "accuracy": accuracy,
            "time": epoch_time,
        }
    )

## Results

In [ ]:
print(f"\nBest accuracy:  {best_accuracy:.2f}%")
print(f"Final accuracy: {training_history[-1]['accuracy']:.2f}%")
print(f"Total time:     {sum(h['time'] for h in training_history):.1f}s")

print(f"\n{'Epoch':>5} | {'Energy':>7} | {'Accuracy':>8} | {'Time':>5}")
print(f"{'-----':>5}-+-{'-------':>7}-+-{'--------':>8}-+-{'-----':>5}")
for h in training_history:
    print(
        f"{h['epoch']:5d} | {h['energy']:7.4f} | {h['accuracy']:7.2f}% | {h['time']:4.1f}s"
    )